# KAEM Profiling
- TPU only (Reactant needs CUDA 12 for GPU, Colab has 11)
- Run cells top to bottom
- Upload `.pb` traces to https://ui.perfetto.dev

In [ ]:
# Cache packages to Google Drive if mounted, else use default ~/.julia
drive_depot = joinpath(ENV["HOME"], "drive", "MyDrive", "julia_depot")
if isdir(dirname(drive_depot))
    mkpath(drive_depot)
    pushfirst!(DEPOT_PATH, drive_depot)
    @info "Package cache: $drive_depot"
end

using Pkg
Pkg.activate("..")
Pkg.instantiate()

In [ ]:
ENV["PERCEPTUAL"] = "false"
ENV["DEVICE"] = "tpu"

using Random, ConfParser, Lux, Reactant, Enzyme, ComponentArrays, Accessors, Statistics
using Printf

include("../src/utils.jl");              using .Utils
include("../src/KAEM/KAEM.jl");          using .KAEM_model, .KAEM_model.EBM_Model, .KAEM_model.GeneratorModel
include("../src/KAEM/model_setup.jl");   using .ModelSetup
include("../src/pipeline/optimizer.jl"); using .optimization
include("../src/pipeline/data_utils.jl"); using .DataUtils: get_vision_dataset
include("../src/KAEM/rng.jl");           using .HLOrng

In [ ]:
function setup(; config="../config/nist_config.ini")
    rng = Random.MersenneTwister(1)
    conf = ConfParse(config); parse_conf!(conf)

    dataset, x_shape, _ = get_vision_dataset("MNIST", 1000, 100, 100; img_resize=(32,32), cnn=false)
    model = init_KAEM(dataset, conf, x_shape; rng=rng)
    x, _ = iterate(model.train_loader)
    x = pu(x)
    opt = create_opt(conf)
    model, opt_st, ps, sk, sl, sr = prep_model(model, x, opt; rng=rng, MLIR=false)
    sr = seed_rand(model; rng=rng)

    return model, opt_st, ps, sk, sl, sr, x
end

function bench(f, args...; warmup=3, trials=20)
    compiled = Reactant.@compile f(args...)
    for _ in 1:warmup; compiled(args...); end
    times = [(@elapsed compiled(args...)) * 1000 for _ in 1:trials]
    return compiled, times
end

function report(items)
    @printf("%-20s %10s %10s %10s\n", "Component", "Med(ms)", "Min(ms)", "Max(ms)")
    println("-"^52)
    for (name, f, args) in items
        _, t = bench(f, args...)
        @printf("%-20s %10.3f %10.3f %10.3f\n", name, median(t), minimum(t), maximum(t))
    end
end

## Benchmark

In [ ]:
model, opt_st, ps, sk, sl, sr, x = setup()
ebm, gen = model.prior, model.lkhood
z = pu(randn(Float32, ebm.q_size, ebm.p_size, model.batch_size))
z_samp = pu(Float32.(Array(first(model.sample_prior(model, ps, sk, sl, sr)))))

report([
    ("ebm_forward",   (z,p,s,l) -> ebm(p,s,l,z),
        (z, ps.ebm, sk.ebm, sl.ebm)),
    ("gen_forward",   (p,s,l,z) -> gen.output_activation(first(gen.generator(p,s,l,z))),
        (ps.gen, sk.gen, sl.gen, z)),
    ("quadrature",    (p,s,l,q) -> ebm.quad(ebm,p,s,l,q),
        (ps.ebm, sk.ebm, sl.ebm, sk.quad)),
    ("inv_transform", (m,p,s,l,r) -> m.sample_prior(m,p,s,l,r),
        (model, ps, sk, sl, sr)),
    ("log_prior",     (z,p,s,l,q) -> model.log_prior(z,ebm,p,s,l,q),
        (z_samp, ps.ebm, sk.ebm, sl.ebm, sk.quad)),
    ("generation",    model,
        (ps, sk, Lux.testmode(sl), sr)),
    ("train_step",    (o,p,s,l,x,i,r) -> model.train_step(o,p,s,l,x,i,r),
        (opt_st, ps, sk, sl, x, 1, sr)),
])

## Profiler Traces
Upload `.pb` files to [Perfetto UI](https://ui.perfetto.dev).

In [ ]:
mkpath("profiles")
step_fn(o,p,s,l,x,i,r) = model.train_step(o,p,s,l,x,i,r)
compiled, _ = bench(step_fn, opt_st, ps, sk, sl, x, 1, sr; warmup=1, trials=1)

Reactant.Profiler.with_profiler("profiles/train_step") do
    for _ in 1:20; compiled(opt_st, ps, sk, sl, x, 1, sr); end
end
println("Saved to profiles/train_step")